# Train the Debate Night scorer add-on

This notebook teaches the game's Standard AI (Qwen 3.5 4B) to judge debate segments the way the
careful verdicts in `training/data/` do. It makes a small add-on file (about 50 MB) that the game
loads next to the model. The game itself never needs this notebook, the internet or any server.

**How to run it (Google Colab, free):**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**. It takes roughly one to two hours.
3. When it finishes, your browser downloads `debate-scorer-lora.gguf`. Send that file to Claude.

On Kaggle instead: turn on **Accelerator → GPU T4** and **Internet**, then **Run all**; the file
appears under **Output**.

In [ ]:
%%capture
# Unsloth makes training this model fit on a free graphics card (same setup as Unsloth's own Qwen 3.5 notebooks).
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [ ]:
# The game's code and the graded debate segments.
import os
os.chdir("/kaggle/working" if os.path.isdir("/kaggle/working") else "/content")
!rm -rf Debate-Night && git clone -q --depth 1 https://github.com/RomeoMonty/Debate-Night
os.chdir("Debate-Night")
!python training/build_labels.py $(for s in training/data/*-segments.jsonl.gz; do echo "$s ${s%-segments.jsonl.gz}-labels.txt"; done) graded.jsonl

In [ ]:
# Training: about an hour on a T4. The loss printed every few steps should fall and then level off.
!python training/train_unsloth.py --scoring graded.jsonl --replay training/data/replay.jsonl.gz --out adapter

In [ ]:
# Turn the add-on into the file the game's engine (llama.cpp b11600) loads.
!rm -rf lc && git clone -q --depth 1 --branch b11600 --filter=blob:none --sparse https://github.com/ggml-org/llama.cpp lc
!cd lc && git sparse-checkout set --no-cone /convert_lora_to_gguf.py /convert_hf_to_gguf.py /conversion /gguf-py
from huggingface_hub import snapshot_download
base = snapshot_download("unsloth/Qwen3.5-4B", allow_patterns=["*.json", "*.txt", "*.jinja", "tokenizer*"])
!python lc/convert_lora_to_gguf.py --base {base} --outtype f16 --outfile debate-scorer-lora.gguf adapter
!ls -la debate-scorer-lora.gguf

In [ ]:
# Hand the file over.
import shutil, os
if os.path.isdir("/kaggle/working"):
    shutil.copy("debate-scorer-lora.gguf", "/kaggle/working/debate-scorer-lora.gguf")
    print("Done: download debate-scorer-lora.gguf from the Output panel and send it to Claude.")
else:
    from google.colab import files
    files.download("debate-scorer-lora.gguf")
    print("Done: your browser is downloading debate-scorer-lora.gguf. Send it to Claude.")